In [112]:
import pandas as pd
from CCA_utils import *

## Baseline Model Panel

In [121]:
study_sovereigns = [
    'Saudi Arabia', 'UAE (Abu Dhabi)', 'Qatar', 'Colombia',
    'Mexico', 'Brazil', 'Egypt', 'Malaysia','Indonesia', 'Philippines', 'Turkey', 'Chile', 'China',
    'South Africa', 'South Korea', 'Thailand']
    
cca_panel_df = pd.read_csv('../data/processed/CCA_V2/CCA_panel.csv')
cca_panel_df['date'] = pd.to_datetime(cca_panel_df['date'])
cca_panel_df = cca_panel_df[cca_panel_df['country'].isin(study_sovereigns)].copy()

T=5.0
vol_window = 12
freq = 'ME'

cca_panel_df.set_index(['date','country'], inplace=True)
cca_panel_df = (
    cca_panel_df
    .groupby('country')
    .resample(freq, level='date')
    .last()
)
cca_panel_df.reset_index(inplace=True)


## Model 1: Specific data

In [122]:
# 1. Prep and Sort (Crucial for merge_asof)
oil_prices = pd.read_csv('../data/processed/Oil/oil_prices_datastream.csv').sort_values('date')
oil_prices['date'] = pd.to_datetime(oil_prices['date'], format='%m/%d/%y')
oil_prices = oil_prices.sort_values('date')

oil_futures = pd.read_csv('../data/processed/Oil/oil_futures.csv').sort_values('date')
oil_futures['date'] = pd.to_datetime(oil_futures['date'], format='%d.%m.%Y')
oil_futures = oil_futures.sort_values('date')

# Ensure your main panel is also sorted by date
cca_panel_df = cca_panel_df.reset_index()
cca_panel_df = cca_panel_df.sort_values('date')

# 2. Perform the Directional Merge
# 'direction="nearest"' finds the closest date, whether it is before or after.
cca_panel_df = pd.merge_asof(
    cca_panel_df, 
    oil_prices[['date', 'Brent']], 
    on='date', 
    direction='nearest'
)

cca_panel_df = pd.merge_asof(
    cca_panel_df, 
    oil_futures[['date', 'Brent_12m']], 
    on='date', 
    direction='nearest'
)

# 5. Restore the Panel Structure and Resample
cca_panel_df.set_index(['date', 'country'], inplace=True)
cca_panel_df = (
    cca_panel_df
    .groupby('country')
    .resample(freq, level='date')
    .last()
)
cca_panel_df.reset_index(inplace=True)
cca_panel_df

,country,date,index,cds_spread,fx_rate,domestic_rate,risk_free_rate,monetary_base_bn_local,external_debt_bn_usd,domestic_debt_bn_local,Brent,Brent_12m
0,Brazil,2014-01-31,0,205.43000,2.412662,0.13400,0.0352,560.240,482.7710,955.827,107.13,101.63
1,Brazil,2014-02-28,1,170.39000,2.338306,0.12700,0.0338,566.324,482.7710,976.705,109.10,104.26
2,Brazil,2014-03-31,2,158.05000,2.261778,0.12700,0.0338,566.324,482.7710,976.705,107.03,103.37
3,Brazil,2014-04-30,3,148.89000,2.244467,0.12800,0.0335,573.521,482.7710,1014.894,107.78,102.63
4,Brazil,2014-05-31,4,139.78000,2.240746,0.12105,0.0312,562.889,482.7710,1043.844,109.56,103.89
...,...,...,...,...,...,...,...,...,...,...,...,...
2107,UAE (Abu Dhabi),2024-08-31,2107,42.14999,3.672825,0.05400,0.0425,734.921,487.2754,26.755,78.89,73.76
2108,UAE (Abu Dhabi),2024-09-30,2108,40.31000,3.672960,0.05400,0.0425,734.921,487.2754,26.755,71.95,70.81
2109,UAE (Abu Dhabi),2024-10-31,2109,45.17000,3.672960,0.04900,0.0410,743.456,487.2754,26.709,73.19,71.08
2110,UAE (Abu Dhabi),2024-11-30,2110,39.78999,3.673095,0.04650,0.0463,747.935,487.2754,26.753,73.20,69.95


In [ ]:
results = pd.DataFrame()
gamma = 0.

for country, group in cca_panel_df.groupby('country'):

    df = group.copy().sort_values('date').reset_index(drop=True)
        
    # --- Unit conversions ---
    r_d = df['domestic_rate']
    r_f = df['risk_free_rate']
    M_bn = df['monetary_base_bn_local']
    dom_D_bn = df['domestic_debt_bn_local']
    ext_D_bn = df['external_debt_bn_usd']
    fx_rate = df['fx_rate']


    # --- LCL$ ---
    df['LCL_usd'] = [
        compute_lcl_usd(m, bd, fx, rd, rf, T)
        for m, bd, fx, rd, rf in zip(
            M_bn, dom_D_bn, fx_rate, r_d, r_f
        )
    ]

        # --- Barrier ---
    df['B_f'] = ext_D_bn
        
    df['basis'] = df['Brent'] / df['Brent_12m']
    df['LCL_aug'] = df['LCL_usd'] * df['basis']
    ann_factor = np.sqrt(52) if freq == 'W' else np.sqrt(12)
    log_ret = np.log(df['LCL_aug'] / df['LCL_aug'].shift(1))



    df['sigma_lcl'] = log_ret.rolling(window=vol_window).std() * ann_factor  # use aug, not base

    # --- Solve CCA with augmented LCL ---
    out = {k: [] for k in ['implied_V', 'implied_sigma_V', 'cca_converged',
                            'distance_to_distress', 'default_prob',
                            'model_spread_bps', 'put_value', 'risky_debt',
                            'leverage']}

    for i, row in df.iterrows():
        cca = solve_CCA(row['LCL_aug'], row['sigma_lcl'], row['B_f'],
                        r_f.iloc[i], T)
        risk = compute_risk(cca['V'], cca['sigma_V'], row['B_f'],
                           r_f.iloc[i], T)

        out['implied_V'].append(cca['V'])
        out['implied_sigma_V'].append(cca['sigma_V'])
        out['cca_converged'].append(cca['converged'])
        out['distance_to_distress'].append(risk['d2'])
        out['default_prob'].append(risk['default_prob'])
        out['model_spread_bps'].append(risk['credit_spread_bps'])
        out['put_value'].append(risk['put_value'])
        out['risky_debt'].append(risk['risky_debt'])
        out['leverage'].append(risk['leverage'])

    for col, vals in out.items():
        df[col] = vals

    results = pd.concat([results, df])


START_DATE = '2015-01-01'
END_DATE = '2024-12-31'

results = results[
    (results['date'] >= START_DATE) & (results['date'] <= END_DATE)
].copy()


In [120]:
results.to_csv("../output/results/M1_results_5YCDS.csv")